In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

proc = Path("data/processed")
try:
    df = pd.read_parquet(proc / "analysis.parquet")
    print("Loaded parquet")
except (FileNotFoundError, ImportError):
    df = pd.read_csv(proc / "analysis.csv")
    print("Loaded CSV")
print(df.shape)                          # expect (69987, 27)

Loaded parquet
(69987, 28)


In [5]:
REF_ORDER = {
    "hba1c_grp":            ["Not measured", "Normal", "High, med changed", "High, med not changed"],
    "age_group":            ["30-60", "<30", ">=60"],
    "sex":                  ["Female", "Male"],
    "race_grp":             ["Caucasian", "AfricanAmerican", "Other", "Missing"],
    "admission_source_grp": ["Emergency room", "Referral", "Other", "Unknown"],
    "admission_type_grp":   ["Emergency", "Urgent", "Elective", "Other/Unknown"],
    "discharge_grp":        ["Home", "Other", "Unknown"],
    "specialty_grp":        ["Internal medicine", "Cardiology", "Family/general practice",
                             "Surgery", "Other", "Missing"],
    "diag1_group":          ["Circulatory", "Respiratory", "Digestive", "Diabetes", "Injury",
                             "Musculoskeletal", "Genitourinary", "Neoplasms", "Other"],
}
for col, levels in REF_ORDER.items():
    df[col] = pd.Categorical(df[col], categories=levels)

In [6]:
# Discharge: Home vs Other (Unknown merged), matching Strack et al.
df["discharge_rep"] = pd.Categorical(
    np.where(df["discharge_grp"] == "Home", "Home", "Other"),
    categories=["Home", "Other"])

# Visit counts capped at the 99th percentile (logistic models only; QC-011)
for col in ["number_inpatient", "number_emergency", "number_outpatient"]:
    cap = df[col].quantile(0.99)
    df[f"{col}_cap"] = df[col].clip(upper=cap)
    print(f"{col}: cap = {cap:g}, rows capped = {(df[col] > cap).sum():,}")

number_inpatient: cap = 3, rows capped = 433
number_emergency: cap = 2, rows capped = 429
number_outpatient: cap = 4, rows capped = 690


In [7]:
# 1. No category turned into NaN: a typo in REF_ORDER would silently do this
print("NaN after setting categories:", int(df[list(REF_ORDER)].isna().sum().sum()))   # 0

# 2. The reference (first level) of each variable
for col in REF_ORDER:
    print(f"{col:22} reference = {df[col].cat.categories[0]}")

# 3. Replication discharge: Other should be 25,667 (22,415 Other + 3,252 Unknown)
print(df["discharge_rep"].value_counts())

NaN after setting categories: 0
hba1c_grp              reference = Not measured
age_group              reference = 30-60
sex                    reference = Female
race_grp               reference = Caucasian
admission_source_grp   reference = Emergency room
admission_type_grp     reference = Emergency
discharge_grp          reference = Home
specialty_grp          reference = Internal medicine
diag1_group            reference = Circulatory
discharge_rep
Home     44320
Other    25667
Name: count, dtype: int64


In [8]:
import statsmodels.formula.api as smf

def or_table(model):
    """Odds ratios with 95% CI and p-values from a fitted logit model."""
    ci = np.exp(model.conf_int())
    t = pd.DataFrame({
        "OR": np.exp(model.params),
        "CI_low": ci[0],
        "CI_high": ci[1],
        "p": model.pvalues,
    }).drop(index="Intercept")
    return t.round({"OR": 2, "CI_low": 2, "CI_high": 2, "p": 4})

In [9]:
m0 = smf.logit("readmit_30 ~ hba1c_grp", data=df).fit(disp=0)
print("Converged:", m0.mle_retvals["converged"], "| N =", int(m0.nobs))
or_table(m0)

Converged: True | N = 69987


,OR,CI_low,CI_high,p
hba1c_grp[T.Normal],0.94,0.84,1.06,0.3256
"hba1c_grp[T.High, med changed]",0.94,0.86,1.04,0.2491
"hba1c_grp[T.High, med not changed]",0.84,0.74,0.95,0.0072


In [10]:
formula = ("readmit_30 ~ hba1c_grp + age_group + sex + race_grp"
           " + admission_source_grp + discharge_rep + diag1_group + specialty_grp"
           " + time_in_hospital + number_inpatient_cap")

m1 = smf.logit(formula, data=df).fit(disp=0)
print("Converged:", m1.mle_retvals["converged"], "| N =", int(m1.nobs))
or_table(m1)

Converged: True | N = 69987


,OR,CI_low,CI_high,p
hba1c_grp[T.Normal],0.95,0.85,1.07,0.4299
"hba1c_grp[T.High, med changed]",0.97,0.87,1.07,0.5134
"hba1c_grp[T.High, med not changed]",0.91,0.80,1.03,0.1445
age_group[T.<30],0.92,0.75,1.13,0.4517
age_group[T.>=60],1.22,1.15,1.30,0.0000
sex[T.Male],1.03,0.97,1.08,0.3150
race_grp[T.AfricanAmerican],1.02,0.95,1.09,0.5970
race_grp[T.Other],0.90,0.79,1.03,0.1337
race_grp[T.Missing],0.86,0.72,1.03,0.0927
admission_source_grp[T.Referral],0.99,0.93,1.06,0.8178


In [11]:
hb = [i for i in or_table(m0).index if i.startswith("hba1c_grp")]
compare = pd.concat({"Unadjusted": or_table(m0).loc[hb], "Adjusted": or_table(m1).loc[hb]}, axis=1)
compare

Unadjusted                        Adjusted  \
                                           OR CI_low CI_high       p       OR   
hba1c_grp[T.Normal]                      0.94   0.84    1.06  0.3256     0.95   
hba1c_grp[T.High, med changed]           0.94   0.86    1.04  0.2491     0.97   
hba1c_grp[T.High, med not changed]       0.84   0.74    0.95  0.0072     0.91   

                                                           
                                   CI_low CI_high       p  
hba1c_grp[T.Normal]                  0.85    1.07  0.4299  
hba1c_grp[T.High, med changed]       0.87    1.07  0.5134  
hba1c_grp[T.High, med not changed]   0.80    1.03  0.1445

In [12]:
from scipy.stats import chi2

m2 = smf.logit(formula.replace("hba1c_grp", "hba1c_grp * diag1_group", 1), data=df).fit(disp=0, maxiter=200)
print("Converged:", m2.mle_retvals["converged"], "| N =", int(m2.nobs))

lr = 2 * (m2.llf - m1.llf)
dof = int(m2.df_model - m1.df_model)
print(f"LR test: chi2 = {lr:.2f}, df = {dof}, p = {chi2.sf(lr, dof):.4f}")

Converged: True | N = 69987
LR test: chi2 = 35.83, df = 24, p = 0.0570


In [13]:
df["hba1c_tested"] = (df["hba1c_grp"] != "Not measured").astype(int)
covars = formula.split("~")[1].replace("hba1c_grp", "hba1c_tested").replace(" + diag1_group", "")

rows = []
for g in df["diag1_group"].cat.categories:
    sub = df[df["diag1_group"] == g]
    m = smf.logit(f"readmit_30 ~ {covars}", data=sub).fit(disp=0, maxiter=200)
    ci = np.exp(m.conf_int().loc["hba1c_tested"])
    rows.append({"diagnosis": g, "n": len(sub), "tested_pct": round(100 * sub["hba1c_tested"].mean(), 1),
                 "OR_tested": round(np.exp(m.params["hba1c_tested"]), 2),
                 "CI_low": round(ci[0], 2), "CI_high": round(ci[1], 2),
                 "p": round(m.pvalues["hba1c_tested"], 4),
                 "converged": m.mle_retvals["converged"]})
pd.DataFrame(rows)

/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,diagnosis,n,tested_pct,OR_tested,CI_low,CI_high,p,converged
0,Circulatory,21389,19.1,1.03,0.92,1.16,0.5639,True
1,Respiratory,9491,19.7,0.77,0.62,0.96,0.0182,True
2,Digestive,6488,13.0,0.98,0.74,1.30,0.9004,True
3,Diabetes,5748,35.9,0.83,0.68,1.03,0.0854,True
4,Injury,4694,12.3,0.60,0.43,0.84,0.0032,True
5,Musculoskeletal,4064,9.9,1.06,0.73,1.53,0.7708,False
6,Genitourinary,3441,15.6,1.19,0.86,1.63,0.2958,True
7,Neoplasms,2538,9.7,1.02,0.65,1.62,0.9223,True
8,Other,12134,18.4,1.04,0.88,1.23,0.6694,True


In [14]:
from sklearn.model_selection import train_test_split

TARGET = "readmit_30"
CAT_FEATURES = ["hba1c_grp", "age_group", "sex", "race_grp", "admission_type_grp",
                "admission_source_grp", "discharge_grp", "specialty_grp", "diag1_group"]
NUM_FEATURES = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                "number_outpatient_cap", "number_emergency_cap", "number_inpatient_cap",
                "number_diagnoses", "n_meds_used_grp", "n_meds_changed_grp",
                "insulin_used", "insulin_adjusted", "any_med_change", "on_diabetes_med"]

X = df[CAT_FEATURES + NUM_FEATURES]
y = df[TARGET]

# First split: 70% train, 30% left over
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
# Second split: the 30% becomes 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=42)

for name, yy in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:10} {len(yy):6,} patients   readmitted {100 * yy.mean():.1f}%")

train      48,990 patients   readmitted 9.0%
validation 10,498 patients   readmitted 9.0%
test       10,499 patients   readmitted 9.0%


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),   # categories -> 0/1 columns
    ("num", StandardScaler(), NUM_FEATURES),                         # numbers -> same scale
])

logreg = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000)),
])

logreg.fit(X_train, y_train)                       # learns from TRAIN only
p_val = logreg.predict_proba(X_val)[:, 1]          # risk score for each VALIDATION patient

print(f"AUROC:  {roc_auc_score(y_val, p_val):.3f}   (coin flip = 0.50)")
print(f"PR-AUC: {average_precision_score(y_val, p_val):.3f}   (baseline = {y_val.mean():.3f})")
print(f"Average predicted risk: {p_val.mean():.3f}   (true rate = {y_val.mean():.3f})")

AUROC:  0.644   (coin flip = 0.50)
PR-AUC: 0.146   (baseline = 0.090)
Average predicted risk: 0.477   (true rate = 0.090)


/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/aljeshbasnet/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features

In [18]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [20]:
from sklearn.ensemble import HistGradientBoostingClassifier

NUM_FEATURES_XGB = [c.replace("_cap", "") for c in NUM_FEATURES]
Xtr = df.loc[X_train.index, CAT_FEATURES + NUM_FEATURES_XGB]
Xva = df.loc[X_val.index,   CAT_FEATURES + NUM_FEATURES_XGB]

prep_gb = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", "passthrough", NUM_FEATURES_XGB),
])

gb = Pipeline([
    ("prep", prep_gb),
    ("model", HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_depth=4,
        class_weight="balanced", random_state=42)),
])

gb.fit(Xtr, y_train)
p_val_gb = gb.predict_proba(Xva)[:, 1]

results = pd.DataFrame({
    "AUROC":  [roc_auc_score(y_val, p_val),  roc_auc_score(y_val, p_val_gb)],
    "PR-AUC": [average_precision_score(y_val, p_val), average_precision_score(y_val, p_val_gb)],
}, index=["Logistic regression", "Gradient boosting"]).round(3)
results

,AUROC,PR-AUC
Logistic regression,0.644,0.146
Gradient boosting,0.644,0.147
